# Практика · ResNet і скіп-зʼєднання

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · ДЗ: [homework.html](homework.html)

> ⏱ **Зошит навчає 38 мереж.** Заміряно: **близько 450 секунд, тобто вісім хвилин**,
> на чотириядерному процесорі в один потік, без відеокарти. Це і є предмет теми —
> деградацію глибоких мереж не показати, не навчивши глибоких мереж. Найдовший розділ —
> восьмий, близько чотирьох хвилин; девʼятий і десятий — ще три разом. Розбір справжньої
> `resnet18` наприкінці миттєвий.

Що зробимо:

1. згенеруємо фігури 28×28 шести класів — той самий датасет, що в блоці 2;
2. зберемо **дві мережі, які відрізняються рівно одним знаком «плюс»**, і переконаємось,
   що в них однакова кількість ваг;
3. порахуємо параметри блоку **руками** й звіримо із `sum(p.numel())`;
4. перевіримо через `torch.allclose`, що блок із нульовим γ **справді тотожний**;
5. заміряємо **норму градієнта по дванадцяти блоках** — головний доказ теми;
6. заміряємо **масштаб активацій** по стосу для чотирьох варіантів блоку;
7. навчимо мережі глибиною 3, 5, 9, 17 і 25 шарів і побудуємо **таблицю деградації**;
8. розберемо справжні `resnet18`, `resnet34`, `resnet50` і `vgg16` через `weights=None`.

## 1 · Середовище

Перша клітинка — не формальність. `torch.set_num_threads(1)` дає дві речі одразу:
швидкість на маленьких мережах (потоки більше домовляються між собою, ніж рахують) і
**детермінованість** — під кількома потоками float-и додаються в іншому порядку, і числа
пливуть від прогону до прогону.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models

# один потік: і швидше на дрібних тензорах, і числа не пливуть від прогону до прогону
torch.set_num_threads(1)

notebook_started = time.perf_counter()

print("torch      :", torch.__version__)
print("numpy      :", np.__version__)
print("потоків CPU:", torch.get_num_threads())

## 2 · Датасет: шість класів фігур 28×28

Той самий генератор, що в блоці 2. Нічого не завантажується: кожна фігура — це формула
плюс гаусів шум. Шість класів означає, що **вгадування навмання дає 0.167** — нижче
цього не буває, і саме з цим числом ми звірятимемо мережі, які зламались.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]


def draw_shape(kind, rng, size=28, jitter=4, noise=0.20):
    """Малює одну фігуру заданого класу як масив 28×28 зі значеннями 0..1."""
    image = np.zeros((size, size), dtype=np.float32)
    # центр зсуваємо, щоб мережа не завчила одне-єдине положення предмета
    center_y = size / 2 + rng.integers(-jitter, jitter + 1)
    center_x = size / 2 + rng.integers(-jitter, jitter + 1)
    radius = rng.integers(5, 9)

    # відстані кожного пікселя від центра — з них складаються всі шість фігур
    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                    # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                  # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                  # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                  # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius) & (distance >= (radius - 3) ** 2)] = 1.0
    elif kind == 4:                                  # хрест
        image[(np.abs(dy) <= 2) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= 2) & (np.abs(dy) <= radius)] = 1.0
    else:                                            # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    image += rng.normal(0, noise, image.shape).astype(np.float32)
    return np.clip(image, 0, 1)


def make_dataset(count, rng):
    """Повертає (count, 1, 28, 28) і (count,). Класи чергуються, тож їх порівну."""
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        slot = i % len(SHAPE_NAMES)
        images[i, 0] = draw_shape(slot, rng)
        labels[i] = slot
    return torch.from_numpy(images), torch.from_numpy(labels)


# зерно 42 — щоб твої числа збіглися з лекцією до останнього знака
rng = np.random.default_rng(42)
train_x, train_y = make_dataset(600, rng)
test_x, test_y = make_dataset(300, rng)

print("навчальна вибірка :", tuple(train_x.shape))
print("перевірочна       :", tuple(test_x.shape))
print("класів            :", len(SHAPE_NAMES), "→ вгадування навмання дає",
      round(1 / len(SHAPE_NAMES), 3))

## 3 · Дві мережі, що відрізняються одним знаком «плюс»

Ось найважливіше місце всієї практики. Блок нижче має прапорець `residual`. Коли він
`False`, блок рахує `y = relu(F(x))`; коли `True` — `y = relu(F(x) + x)`.

**Більше не відрізняється нічого**: та сама кількість згорток, батчнормів, ReLU і ваг.
Якби мережі відрізнялись ще чимось, порівняння нічого не вартувало б.

In [ ]:
class Block(nn.Module):
    """Дві згортки 3×3 з батчнормом. residual=True вмикає скіп-зʼєднання."""

    def __init__(self, channels, residual, zero_init=False):
        super().__init__()
        # bias=False, бо батчнорм одразу за згорткою однаково відніме будь-який зсув
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()
        self.residual = residual
        if zero_init:
            # γ другого BN у нуль: тоді F(x) = 0 і блок на старті нічого не робить
            nn.init.zeros_(self.bn2.weight)

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.residual:
            out = out + x          # ← оцей рядок і є вся різниця між двома мережами
        return self.relu(out)      # ReLU стоїть ПІСЛЯ додавання, і це навмисно


class Net(nn.Module):
    """Стем → n_blocks однакових блоків → глобальне усереднення → лінійний шар.

    Глибина в згорткових шарах = 1 (стем) + 2 на кожен блок.
    """

    def __init__(self, n_blocks, residual, zero_init=False, channels=16, n_classes=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, channels, 3, stride=2, padding=1, bias=False),  # 28×28 → 14×14
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.MaxPool2d(2),                                             # 14×14 → 7×7
        )
        self.blocks = nn.Sequential(*[
            Block(channels, residual, zero_init) for _ in range(n_blocks)
        ])
        self.pool = nn.AdaptiveAvgPool2d(1)     # глобальне усереднення з теми 08
        self.fc = nn.Linear(channels, n_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


print("глибина | параметрів проста | параметрів залишкова | однаково?")
for n_blocks in [1, 2, 4, 8, 12]:
    depth = 1 + 2 * n_blocks
    torch.manual_seed(0)
    plain = count_params(Net(n_blocks, residual=False))
    torch.manual_seed(0)
    residual = count_params(Net(n_blocks, residual=True))
    print(f"{depth:7d} | {plain:17d} | {residual:20d} | {plain == residual}")
    assert plain == residual, "мережі мають різну кількість ваг — порівняння нечесне!"
print("\n✅ на кожній глибині обидві мережі мають однакову кількість ваг")

## 4 · Параметри блоку руками проти `sum(p.numel())`

Найкорисніша звичка курсу: перш ніж вірити бібліотеці, порахувати самому.

Блок на 16 каналів:

- дві згортки 3×3 без зсуву: кожна має `16 × 16 × 3 × 3` ваг;
- два батчнорми: кожен тримає γ і β **на канал**, тобто `2 × 16` чисел.

Скіп-зʼєднання не додає нічого: це операція додавання, а не шар із вагами.

In [ ]:
channels = 16

conv_params_by_hand = 2 * (channels * channels * 3 * 3)
bn_params_by_hand = 2 * (2 * channels)
total_by_hand = conv_params_by_hand + bn_params_by_hand

torch.manual_seed(0)
block_with_skip = Block(channels, residual=True)
torch.manual_seed(0)
block_without_skip = Block(channels, residual=False)

print(f"дві згортки 3×3: 2 × {channels}×{channels}×3×3 = {conv_params_by_hand}")
print(f"два батчнорми  : 2 × 2×{channels} = {bn_params_by_hand}")
print(f"разом руками   : {total_by_hand}")
print(f"sum(p.numel()) : {count_params(block_with_skip)}")
print(f"блок без скіпа : {count_params(block_without_skip)}")

assert count_params(block_with_skip) == total_by_hand, "розрахунок розійшовся!"
assert count_params(block_without_skip) == total_by_hand, "скіп чомусь щось коштує!"
print("\n✅ збігається — і скіп-зʼєднання справді не має жодної ваги")

## 5 · Чи справді блок із нульовим γ тотожний

Теорія каже: якщо γ останнього батчнорму дорівнює нулю, то `F(x) = 0`, і блок віддає
`relu(0 + x)`. А `x` прийшов із попереднього ReLU, тобто вже невідʼємний — отже,
`relu(x) = x`.

Перевіримо це не на віру, а `torch.allclose`. Вхід беремо невідʼємний саме тому, що в
мережі блок стоїть після ReLU і іншого входу не бачить.

In [ ]:
torch.manual_seed(0)
zero_block = Block(16, residual=True, zero_init=True).eval()
torch.manual_seed(0)
usual_block = Block(16, residual=True, zero_init=False).eval()

# вхід невідʼємний: у мережі блок стоїть одразу після ReLU
probe_input = torch.relu(torch.randn(4, 16, 7, 7))

zero_is_identity = torch.allclose(zero_block(probe_input), probe_input, atol=1e-6)
usual_is_identity = torch.allclose(usual_block(probe_input), probe_input, atol=1e-6)
usual_max_deviation = (usual_block(probe_input) - probe_input).abs().max().item()

print("блок із γ = 0 : allclose(block(x), x) =", zero_is_identity)
print("звичайний блок: allclose(block(x), x) =", usual_is_identity)
print(f"звичайний блок: максимальне відхилення виходу від входу = {usual_max_deviation:.2f}")

assert zero_is_identity, "блок із нульовим γ мав би бути тотожним!"
assert not usual_is_identity, "звичайний блок тотожним бути не мав!"
print("\n✅ нульова ініціалізація справді робить блок тотожним на старті")

## 6 · Норма градієнта по шарах — головний доказ теми

Беремо мережу з дванадцяти блоків (25 згорткових шарів), подаємо один батч із 64
зображень, рахуємо помилку й робимо зворотний прохід. Далі дивимось на **норму
градієнта** ваг першої згортки в кожному блоці.

Норма — це довжина вектора: корінь із суми квадратів усіх чисел. Одне число, що каже
«наскільки сильно цей шар зараз штовхають».

⚠️ Підручники обіцяють згасання градієнта до входу. Подивись, що вийде насправді.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
batch_x, batch_y = train_x[:64], train_y[:64]


def gradient_profile(residual, zero_init=False, n_blocks=12):
    """Норма градієнта conv1.weight у кожному блоці — на старті, до першого кроку."""
    torch.manual_seed(0)
    model = Net(n_blocks, residual, zero_init)
    model.train()
    loss_fn(model(batch_x), batch_y).backward()
    return [block.conv1.weight.grad.norm().item() for block in model.blocks]


grad_plain = gradient_profile(residual=False)
grad_residual = gradient_profile(residual=True)
grad_zero = gradient_profile(residual=True, zero_init=True)

print("блок |     проста |  залишкова | залишкова з γ = 0")
for i in range(12):
    print(f"{i + 1:4d} | {grad_plain[i]:10.4f} | {grad_residual[i]:10.4f} | {grad_zero[i]:10.4f}")

print()
print("Розкид між першим блоком (біля входу) і останнім (біля виходу):")
print(f"  проста    : {grad_plain[0] / grad_plain[-1]:.1f}×")
print(f"  залишкова : {grad_residual[0] / grad_residual[-1]:.1f}×")
print(f"  з γ = 0   : сума градієнтів по всіх блоках = {sum(grad_zero):.1f}")

**Що ми щойно побачили.** У простій мережі градієнт до входу не згасає, а
**розростається** — у пʼятдесят разів. Причина — батчнорм: він ділить сигнал на його ж
стандартне відхилення, і на зворотному проході це ділення стає множенням. Двадцять
чотири такі множення поспіль дають накопичений розгін.

Висновок від цього не змінюється: сигнал по шарах **нерівний**, а один крок оптимізатора
має бути доречним одразу для всіх шарів. У залишковій мережі розкид у дванадцять разів
менший.

Окремо зверни увагу на третю колонку: при γ = 0 градієнт усередині блоків дорівнює
**рівно нулю**. Блоки на першому кроці не вчаться взагалі.

## 7 · Масштаб активацій по стосу — чотири варіанти блоку

Тепер прямий прохід. Дивимось на **середнє значення виходу** кожного блоку на старті
навчання. Порівнюємо чотири проводки:

- **без скіпа** — батчнорм тримає масштаб силою, тож він рівний;
- **скіп перед останньою ReLU** — канонічний ResNet;
- **скіп після останньої ReLU** — так робити не можна, і зараз буде видно чому;
- **скіп перед ReLU з нульовим γ** — блок тотожний, масштаб не рухається взагалі.

In [ ]:
class BlockAfterRelu(nn.Module):
    """Скіп додається ПІСЛЯ останнього ReLU. Так у ResNet не роблять."""

    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))   # ReLU до додавання
        return out + x                               # гілка вміє лише додавати додатне


def activation_profile(make_model, n_blocks=12):
    """Середнє значення виходу кожного блоку на старті навчання."""
    torch.manual_seed(0)
    model = make_model()
    means = []
    for block in model.blocks:
        block.register_forward_hook(
            lambda module, inputs, output, store=means: store.append(output.mean().item()))
    model.train()
    model(batch_x)
    return means


def make_after_relu_net():
    model = Net(12, residual=True)
    model.blocks = nn.Sequential(*[BlockAfterRelu(16) for _ in range(12)])
    return model


act_plain = activation_profile(lambda: Net(12, residual=False))
act_residual = activation_profile(lambda: Net(12, residual=True))
act_zero = activation_profile(lambda: Net(12, residual=True, zero_init=True))
act_after = activation_profile(make_after_relu_net)

print("блок | без скіпа | скіп до ReLU | скіп після ReLU | скіп, γ = 0")
for i in range(12):
    print(f"{i + 1:4d} | {act_plain[i]:9.2f} | {act_residual[i]:12.2f} | "
          f"{act_after[i]:15.2f} | {act_zero[i]:11.2f}")

print()
print("Розгін від першого блоку до дванадцятого:")
for name, profile in [("без скіпа", act_plain), ("скіп до ReLU", act_residual),
                      ("скіп після ReLU", act_after), ("скіп, γ = 0", act_zero)]:
    print(f"  {name:16s}: {profile[0]:.2f} → {profile[11]:.2f}  "
          f"(у {profile[11] / profile[0]:.2f} раза)")

## 8 · Навчання: таблиця деградації

Ось головний дослід теми, і він найдовший — близько чотирьох хвилин.

Умови однакові для всіх: SGD із моментом 0.9, швидкість навчання 0.005, двадцять епох,
батч 64. Кожна клітинка таблиці — **середнє з трьох прогонів** із різними зернами. Один
прогін на такій маленькій задачі занадто шумний, щоб на нього спиратись: ми це
перевірили, і розкид сягав 0.3.

In [ ]:
def accuracy(model, x, y, batch=200):
    """Частка правильних відповідей. Обовʼязково в режимі eval() — див. тему 11."""
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(x), batch):
            predicted = model(x[i:i + batch]).argmax(1)
            correct += (predicted == y[i:i + batch]).sum().item()
    model.train()
    return correct / len(x)


def train_model(model, epochs=20, lr=0.005, batch=64, seed=0):
    """Навчає модель і повертає (навчальна точність, перевірочна точність)."""
    torch.manual_seed(seed)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    n = len(train_x)
    # окремий генератор для перемішування, щоб порядок батчів не залежав від ваг
    order_gen = torch.Generator().manual_seed(seed)
    model.train()
    for _ in range(epochs):
        order = torch.randperm(n, generator=order_gen)
        for i in range(0, n, batch):
            index = order[i:i + batch]
            optimizer.zero_grad()
            loss = loss_fn(model(train_x[index]), train_y[index])
            loss.backward()
            optimizer.step()
    return accuracy(model, train_x, train_y), accuracy(model, test_x, test_y)


def average_over_seeds(make_model, seeds=(0, 1, 2), **kwargs):
    """Три прогони з різними зернами — і середнє. Повертає ще й розкид."""
    train_scores, test_scores = [], []
    for seed in seeds:
        torch.manual_seed(seed)
        train_accuracy, test_accuracy = train_model(make_model(), seed=seed, **kwargs)
        train_scores.append(train_accuracy)
        test_scores.append(test_accuracy)
    return (float(np.mean(train_scores)), float(np.mean(test_scores)),
            min(test_scores), max(test_scores))


started = time.perf_counter()
depth_table = []
print("глибина | параметрів | проста навч | проста тест | залишк. навч | залишк. тест")
for n_blocks in [1, 2, 4, 8, 12]:
    depth = 1 + 2 * n_blocks
    torch.manual_seed(0)
    n_weights = count_params(Net(n_blocks, residual=False))
    plain = average_over_seeds(lambda nb=n_blocks: Net(nb, residual=False))
    resid = average_over_seeds(lambda nb=n_blocks: Net(nb, residual=True))
    depth_table.append((depth, n_weights, plain, resid))
    print(f"{depth:7d} | {n_weights:10d} | {plain[0]:11.3f} | {plain[1]:11.3f} | "
          f"{resid[0]:12.3f} | {resid[1]:12.3f}")
print(f"\nзаміряно за {time.perf_counter() - started:.0f} с")

### Читаємо таблицю

**Ліва половина — проста мережа.** Знайди рядок із найбільшою навчальною точністю. Далі
кожні додані шари роблять гірше, і на 25 шарах мережа з вшестеро більшою кількістю ваг
вивчила навчальні дані **гірше** за пʼятишарову.

**І це не перенавчання.** Подивись на 25 шарів: навчальна й перевірочна точності майже
однакові. Перенавчання виглядає інакше — висока навчальна, низька перевірочна, великий
розрив між ними. Тут розриву практично немає: мережа не завчила дані, вона їх не
вивчила.

**Права половина — залишкова мережа.** Ті самі ваги, той самий оптимізатор, той самий
датасет. Різниця — рядок `out = out + x`.

In [ ]:
print("Розрив між навчальною й перевірочною точністю — підпис перенавчання:")
print("глибина | проста: навч − тест | залишкова: навч − тест")
for depth, _, plain, resid in depth_table:
    print(f"{depth:7d} | {plain[0] - plain[1]:19.3f} | {resid[0] - resid[1]:22.3f}")

deepest_depth, _, deepest_plain, deepest_resid = depth_table[-1]
best_plain = max(row[2][0] for row in depth_table)
print()
print(f"Найкраща навчальна точність простої мережі за всю таблицю: {best_plain:.3f}")
print(f"Вона ж на {deepest_depth} шарах: {deepest_plain[0]:.3f} "
      f"(борг {best_plain - deepest_plain[0]:.3f})")
print(f"Залишкова на {deepest_depth} шарах: {deepest_resid[0]:.3f}")

## 9 · Три варіанти блоку на глибині 25

Тепер порівняємо на найглибшій мережі три проводки скіпа й канонічний варіант із
нульовою ініціалізацією. Кожен — три прогони, як і раніше.

Тут буде **несподіванка**, і ми її не ховатимемо: нульова ініціалізація, яку радять усі
посібники з ResNet, на нашій задачі програє з великим відривом. Чому — розберемось у
наступній клітинці.

In [ ]:
started = time.perf_counter()
variants = [
    ("проста (без скіпа)", lambda: Net(12, residual=False)),
    ("скіп до ReLU", lambda: Net(12, residual=True)),
    ("скіп після ReLU", make_after_relu_net),
    ("скіп, γ = 0", lambda: Net(12, residual=True, zero_init=True)),
]

deep_results = {}
print("варіант             | навч  | тест  | тест: мінімум-максимум")
for name, make_model in variants:
    train_accuracy, test_accuracy, low, high = average_over_seeds(make_model)
    deep_results[name] = (train_accuracy, test_accuracy)
    print(f"{name:19s} | {train_accuracy:.3f} | {test_accuracy:.3f} | {low:.3f}-{high:.3f}")
print(f"\nзаміряно за {time.perf_counter() - started:.0f} с")

## 10 · Чому нульова ініціалізація програла

Гіпотеза проста: при γ = 0 градієнт усередині блоків дорівнює нулю (ми це бачили в
розділі 6), тож блоки «вмикаються» лише в міру того, як сама γ відходить від нуля.
Якщо вона рухається повільно, за двадцять епох блоки просто не встигають нічого вивчити.

Перевіримо двома способами: подивимось, куди дійшла γ за двадцять епох, і дамо мережі
**шістдесят** епох замість двадцяти.

In [ ]:
started = time.perf_counter()

torch.manual_seed(0)
zero_net = Net(12, residual=True, zero_init=True)
gamma_before = [block.bn2.weight.abs().mean().item() for block in zero_net.blocks]
train_model(zero_net, epochs=20, lr=0.005, seed=0)
gamma_after = [block.bn2.weight.abs().mean().item() for block in zero_net.blocks]

print("Середнє |γ| другого батчнорму по блоках:")
print("  до навчання :", " ".join(f"{v:.3f}" for v in gamma_before))
print("  після 20 епох:", " ".join(f"{v:.3f}" for v in gamma_after))
print(f"  у середньому по стосу: {np.mean(gamma_after):.3f}")

print("\nТой самий дослід на 60 епохах замість 20 (один прогін, зерно 0):")
for name, make_model in [("залишкова", lambda: Net(12, residual=True)),
                         ("залишкова, γ = 0", lambda: Net(12, residual=True, zero_init=True))]:
    torch.manual_seed(0)
    train_accuracy, test_accuracy = train_model(make_model(), epochs=60, lr=0.005, seed=0)
    print(f"  {name:17s}: навч {train_accuracy:.3f}  тест {test_accuracy:.3f}")
print(f"\nзаміряно за {time.perf_counter() - started:.0f} с")

**Висновок, який варто запамʼятати.** Нульова ініціалізація — не безкоштовний трюк, а
**прийом для довгого розкладу**. Вона тримає масштаб сигналу рівним і робить глибокий
стос керованим, але ціну бере одразу, а віддає лише тоді, коли епох досить, щоб γ
встигла відійти від нуля. У статті ResNet її застосовували на ImageNet із дев'яноста
епохами.

Не переноси це правило на свої задачі наосліп — заміряй, як ми щойно.

## 11 · Справжні архітектури через `weights=None`

Далі — жодного навчання, лише розбір. Виклик `models.resnet18(weights=None)` будує
архітектуру **локально**: створює всі шари, розставляє форми, ініціалізує ваги
випадковими числами — і **нічого не завантажує з мережі**.

In [ ]:
def describe(name, model):
    """Друкує, скільки ваг у згортках, скільки в лінійних шарах, скільки в батчнормах."""
    total = sum(p.numel() for p in model.parameters())
    in_conv = sum(p.numel() for m in model.modules()
                  if isinstance(m, nn.Conv2d) for p in m.parameters())
    in_linear = sum(p.numel() for m in model.modules()
                    if isinstance(m, nn.Linear) for p in m.parameters())
    in_bn = sum(p.numel() for m in model.modules()
                if isinstance(m, nn.BatchNorm2d) for p in m.parameters())
    n_conv = sum(1 for m in model.modules() if isinstance(m, nn.Conv2d))
    print(f"{name:10s} | {total:11,} | {n_conv:3d} згорток | "
          f"згортки {in_conv / total * 100:5.1f} % | голова {in_linear / total * 100:5.1f} % | "
          f"BN {in_bn / total * 100:4.2f} %")


resnet18 = models.resnet18(weights=None)
resnet34 = models.resnet34(weights=None)
resnet50 = models.resnet50(weights=None)
vgg16 = models.vgg16(weights=None)

print("мережа     |    усього ваг | згорткових  | де саме лежать ваги")
for name, model in [("AlexNet", models.alexnet(weights=None)), ("VGG16", vgg16),
                    ("ResNet18", resnet18), ("ResNet34", resnet34), ("ResNet50", resnet50)]:
    describe(name, model)

print()
vgg_head = sum(p.numel() for p in vgg16.classifier.parameters())
resnet_head = sum(p.numel() for p in resnet18.fc.parameters())
resnet18_bn = sum(p.numel() for m in resnet18.modules()
                  if isinstance(m, nn.BatchNorm2d) for p in m.parameters())
print(f"голова VGG16    : {vgg_head:,} ваг (три повнозвʼязні шари)")
print(f"голова ResNet18 : {resnet_head:,} ваг (один лінійний шар)")
print(f"співвідношення  : {vgg_head / resnet_head:.0f} : 1")
print(f"\nусі батчнорми ResNet18 разом: {resnet18_bn:,} ваг — шар, без якого глибока")
print("мережа не навчається взагалі, важить менше однієї тисячної від неї")

### Будова resnet18 по стадіях

Чотири стадії по два блоки. На початку кожної стадії, крім першої, сторона карти
зменшується вдвічі, а кількість каналів подвоюється — саме там скіпу й потрібна
проєкція 1×1.

In [ ]:
print("стадія | блоків | канали     | крок першого блоку")
for stage_name in ["layer1", "layer2", "layer3", "layer4"]:
    stage = getattr(resnet18, stage_name)
    print(f"{stage_name:6s} | {len(stage):6d} | "
          f"{stage[0].conv1.in_channels:3d} → {stage[-1].conv2.out_channels:3d} | "
          f"{stage[0].conv1.stride[0]}")

print()
print("Скіп-зʼєднання з проєкцією:")
projection_total = 0
projection_count = 0
for name, module in resnet18.named_modules():
    if name.endswith("downsample") and len(list(module.children())) > 0:
        block = resnet18.get_submodule(name.rsplit(".", 1)[0])
        projection = sum(p.numel() for p in module.parameters())
        block_total = sum(p.numel() for p in block.parameters())
        conv = module[0]
        print(f"  {name:22s} {conv.in_channels:3d} → {conv.out_channels:3d}, "
              f"ядро 1×1, крок {conv.stride[0]}: {projection:>7,} ваг "
              f"= {projection / block_total * 100:.2f} % блоку")
        projection_total += projection
        projection_count += 1

resnet18_total = sum(p.numel() for p in resnet18.parameters())
n_blocks_total = sum(len(getattr(resnet18, s)) for s in ["layer1", "layer2", "layer3", "layer4"])
print(f"\n  проєкцій {projection_count} із {n_blocks_total} блоків, разом {projection_total:,} ваг "
      f"= {projection_total / resnet18_total * 100:.2f} % мережі")
print(f"  решта {n_blocks_total - projection_count} скіпів не коштує нічого")

## 12 · Пляшкове горло: скільки воно економить

Звичайний блок — дві згортки 3×3, тобто `2 · c² · 9` ваг. Пляшкове горло стискає канали
вчетверо, робить просторову роботу на вужчому місці й розтискає назад.

Порахуємо обидва варіанти формулою й перевіримо на справжньому блоці з `torchvision`.

In [ ]:
from torchvision.models.resnet import BasicBlock, Bottleneck

SIDE = 14      # сторона карти ознак, для якої рахуємо множення

print("каналів |  звичайний блок |  пляшкове горло |     звич. множень |    горло множень | виграш")
for c in [64, 128, 256, 512, 1024, 2048]:
    narrow = c // 4
    basic_params = 2 * c * c * 9 + 4 * c
    bottleneck_params = (c * narrow + 2 * narrow) + (narrow * narrow * 9 + 2 * narrow) \
                        + (narrow * c + 2 * c)
    basic_macs = 2 * c * c * 9 * SIDE * SIDE
    bottleneck_macs = (c * narrow + narrow * narrow * 9 + narrow * c) * SIDE * SIDE
    print(f"{c:7d} | {basic_params:15,} | {bottleneck_params:15,} | "
          f"{basic_macs:17,} | {bottleneck_macs:16,} | {basic_params / bottleneck_params:5.1f}×")

# звіряємо формулу з тим, що будує torchvision
reference_basic = sum(p.numel() for p in BasicBlock(256, 256).parameters())
reference_bottleneck = sum(p.numel() for p in Bottleneck(256, 64).parameters())
formula_basic = 2 * 256 * 256 * 9 + 4 * 256
print()
print(f"BasicBlock(256, 256) за формулою : {formula_basic:,}")
print(f"BasicBlock(256, 256) з torchvision: {reference_basic:,}")
assert formula_basic == reference_basic, "формула розійшлася з бібліотекою!"
print("✅ формула збігається з torchvision")

print(f"\nBottleneck(256, 64) з torchvision: {reference_bottleneck:,} ваг "
      f"(вхід і вихід ті самі 256 каналів)")

In [ ]:
# Найширша стадія resnet50 — і скільки вона важила б на звичайних блоках
real_layer4 = sum(p.numel() for p in resnet50.layer4.parameters())
imaginary_layer4 = sum(p.numel() for p in nn.Sequential(
    BasicBlock(2048, 2048), BasicBlock(2048, 2048), BasicBlock(2048, 2048)).parameters())

print(f"layer4 resnet50, як воно є (три пляшкові горла): {real_layer4:>12,} ваг")
print(f"той самий layer4 на звичайних блоках           : {imaginary_layer4:>12,} ваг")
print(f"різниця: у {imaginary_layer4 / real_layer4:.1f} раза")
print(f"\nдля порівняння, уся VGG16: {sum(p.numel() for p in vgg16.parameters()):,} ваг")

In [ ]:
print(f"⏱ зошит виконався за {time.perf_counter() - notebook_started:.0f} с")

## Завдання

### 🟢 Рівень 1

Візьми таблицю з розділу 8 і додай до неї глибину **33 шари** (16 блоків). Чи продовжує
проста мережа падати? Чи тримається залишкова?

**Зроблено, якщо:** у таблиці зʼявився рядок «33», і ти можеш назвати числом, наскільки
навчальна точність простої мережі на 33 шарах відрізняється від її найкращої.

### 🟡 Рівень 2

Повтори замір норми градієнта з розділу 6 **після** десяти епох навчання, а не на старті.
Порівняй із тим, що було на старті: розкид виріс, зменшився чи лишився таким самим?

**Зроблено, якщо:** ти надрукував обидва профілі поруч і назвав відношення «перший блок /
останній» для чотирьох випадків: проста на старті, проста після навчання, залишкова на
старті, залишкова після навчання.

### 🔴 Рівень 3

Збери блок із **пляшковим горлом** на нашому датасеті: `1×1` стискає 16 каналів до 4,
`3×3` працює на чотирьох, `1×1` розтискає назад до 16. Постав скіп-зʼєднання так само, як
у звичайному блоці.

Навчи мережу з дванадцяти таких блоків і порівняй із мережею зі звичайних блоків за
трьома числами: параметри, час навчання, перевірочна точність.

**Зроблено, якщо:** ти надрукував таблицю з трьох рядків і сказав, чи виправдовує себе
пляшкове горло на шістнадцяти каналах — і чому відповідь тут інша, ніж на 2048.